# API geography benchmark

Compares the TaxonWorks API calls used by the identification-key geography-scope
feature (`modules/keys/composables/useKeyGeography.js`): the `dwc` inventory route,
the raw `/dwc_occurrences` endpoint, and `/asserted_distributions` with and without
`descendants=true`.

Run with the **Python 3 (ipykernel)** kernel (this repo's `weevil-hostplants` conda
env: `requests`, `pandas`, `pyyaml` already installed there). Pick it from the kernel
selector if a different kernel is active.

Reads the API token from `config/api.yml` (two levels up from this notebook) &mdash;
nothing is hard-coded here.

In [ ]:
import time
from pathlib import Path

import pandas as pd
import requests
import yaml

CONFIG_PATH = Path("../../config/api.yml")
cfg = yaml.safe_load(CONFIG_PATH.read_text())
BASE_URL = cfg["url"]
TOKEN = cfg["project_token"]

session = requests.Session()


def timed_get(path, params=None, timeout=300):
    """GET path with the project token, return timing/size/row-count/data."""
    params = dict(params or {})
    params["project_token"] = TOKEN
    t0 = time.perf_counter()
    r = session.get(f"{BASE_URL}{path}", params=params, timeout=timeout)
    elapsed = time.perf_counter() - t0
    try:
        data = r.json()
        rows = len(data) if isinstance(data, list) else None
    except ValueError:
        data, rows = None, None
    params.pop("project_token", None)
    return {
        "path": path,
        "params": params,
        "status": r.status_code,
        "seconds": round(elapsed, 2),
        "bytes": len(r.content),
        "rows": rows,
        "data": data,
    }


results = []


def run(label, path, params=None, timeout=300):
    """timed_get + record it in `results` + print a one-line summary."""
    r = timed_get(path, params, timeout=timeout)
    r["label"] = label
    results.append(r)
    print(f"{label}: {r['seconds']}s, {r['bytes']:,} bytes, rows={r['rows']}, HTTP {r['status']}")
    return r


def table():
    if not results:
        return pd.DataFrame(columns=["label", "path", "params", "seconds", "bytes", "rows", "status"])
    return pd.DataFrame(results)[["label", "path", "params", "seconds", "bytes", "rows", "status"]]

## Resolve a taxon

`resolve_taxon("Name")` looks up `taxon_name_id`, `otu_id` and `rank` for any valid
name so the benchmark cells below can be pointed at whatever taxon you want to try,
not just the two used as running examples in this session.

In [ ]:
def resolve_taxon(name):
    matches = timed_get("/taxon_names", {"name": name})["data"] or []
    matches = [t for t in matches if t["name"] == name] or matches
    if not matches:
        raise ValueError(f"no taxon_name found for {name!r}")
    tn = matches[0]
    otus = timed_get("/otus", {"taxon_name_id[]": tn["id"]})["data"] or []
    return {
        "name": name,
        "taxon_name_id": tn["id"],
        "otu_id": otus[0]["id"] if otus else None,
        "rank": tn["rank"].rsplit("::", 1)[-1],
    }


# Running examples from this session:
#   Curculionoidea = superfamily, the slow case (key 5024)
#   Adosomus       = genus, the fast control case
curculionoidea = resolve_taxon("Curculionoidea")
adosomus = resolve_taxon("Adosomus")
curculionoidea, adosomus

## Benchmark suite

Five call shapes, all paginated at the same `per`/`page` so they're comparable:

1. `/otus/:id/inventory/dwc.json` &mdash; **unpaginated** (`page`/`per` omitted). This is
   what the key code called before this session's fix; per the API docs
   (`docs/openapi/otu.yaml`), omitting `page`/`per` returns *every* row in one response.
   **Opt-in only** (`include_unpaginated=True`) &mdash; on a superfamily this took 2m48s /
   107 MB in this session, so don't fire it by accident.
2. `/otus/:id/inventory/dwc.json` &mdash; paginated.
3. `/dwc_occurrences?taxon_name_id[]=` &mdash; the same underlying query as #2
   (`DwcOccurrence.scoped_by_otu` calls this filter internally) called directly.
4. `/asserted_distributions?otu_id[]=` &mdash; distributions asserted directly on the
   OTU, no descendant walk.
5. `/asserted_distributions?taxon_name_id[]=&descendants=true` &mdash; the rollup used
   for genus-and-above terminals; what `useKeyGeography.js` step 2 already calls.

In [ ]:
def benchmark(taxon, per=1000, page=1, include_unpaginated=False):
    prefix = f"{taxon['name']} ({taxon['rank']})"

    if include_unpaginated:
        run(f"{prefix}: dwc.json UNPAGINATED", f"/otus/{taxon['otu_id']}/inventory/dwc.json")

    run(
        f"{prefix}: dwc.json paginated",
        f"/otus/{taxon['otu_id']}/inventory/dwc.json",
        {"per": per, "page": page},
    )
    run(
        f"{prefix}: dwc_occurrences direct",
        "/dwc_occurrences",
        {"taxon_name_id[]": taxon["taxon_name_id"], "per": per, "page": page},
    )
    run(
        f"{prefix}: asserted_distributions otu_id",
        "/asserted_distributions",
        {"otu_id[]": taxon["otu_id"], "per": per, "page": page},
    )
    run(
        f"{prefix}: asserted_distributions descendants",
        "/asserted_distributions",
        {"taxon_name_id[]": taxon["taxon_name_id"], "descendants": "true", "per": per, "page": page},
    )

In [ ]:
# Fast control case first — genus rank, nothing here should be slow.
benchmark(adosomus)
table()

In [ ]:
# The slow case (key 5024). Paginated calls only by default — pass
# include_unpaginated=True below if you deliberately want to re-time the 2m48s/107MB call.
benchmark(curculionoidea)
table()

## Page-count probe

`per`/`page` shrinks the response, but for a fixed-cost query (see the `dwc.json`
results above — `per=1` costs the same as `per=1000`) the number of pages you'd need
to fetch everything is what actually matters for a real key. `count_pages` binary-
searches for the last non-empty page so you can compare "pages needed" across
endpoints without fetching all of them.

In [ ]:
def count_pages(path, params, per=1000, hi_start=2, max_hi=4096):
    """Binary search for the last page with rows == per (i.e. still full).
    Returns (last_full_page, seconds_for_that_probe_page). Cheap: O(log n) requests.
    """
    base = dict(params or {})
    base["per"] = per

    def rows_at(page):
        r = timed_get(path, {**base, "page": page})
        return len(r["data"] or []), r["seconds"]

    lo = 1
    hi = hi_start
    lo_n, lo_s = rows_at(lo)
    if lo_n < per:
        return lo if lo_n else 0, lo_s
    while True:
        n, s = rows_at(hi)
        if n < per or hi >= max_hi:
            break
        lo, hi = hi, hi * 2
    # lo is full, hi is short (or capped) — binary search between them
    last_seconds = s
    while hi - lo > 1:
        mid = (lo + hi) // 2
        n, s = rows_at(mid)
        last_seconds = s
        if n == per:
            lo = mid
        else:
            hi = mid
    return lo, last_seconds


pages, secs = count_pages(
    "/asserted_distributions",
    {"taxon_name_id[]": curculionoidea["taxon_name_id"], "descendants": "true"},
)
print(f"asserted_distributions descendants=true, {curculionoidea['name']}: ~{pages} full pages of 1000 (last probe {secs}s)")

## Findings so far (2026-09-04 session)

Recorded here so re-running this notebook isn't the only record of them.

| Call | Curculionoidea (superfamily, OTU 708185) | Adosomus (genus, OTU 732685) |
|---|---|---|
| `dwc.json` unpaginated | 107 MB, **2m 48s** | 108 KB, 1.1s |
| `dwc.json` `per=1000&page=1` | 1.28 MB, **39.9s** | — |
| `dwc.json` `per=100&page=1` | 124 KB, **38.6s** | — |
| `dwc.json` `per=1&page=1` | 1.2 KB, **37.7s** | — |
| `/dwc_occurrences?taxon_name_id[]=` `per=1000&page=1` | 1.24 MB, **39.2s** | — |
| `/asserted_distributions?taxon_name_id[]=&descendants=true` `per=1000` | 1.9 MB, **4.5–5.4s per page**, ~50–54 pages total | — |

**Conclusion:** the `dwc.json` / `dwc_occurrences` cost is a fixed ~38–40s *per request*
for a superfamily-scale `taxon_name_id`, independent of `per` — it's the cost of
computing the matching set (self + descendants + synonyms, unioned across
AssertedDistribution/CollectionObject/FieldOccurrence) before any `LIMIT` is applied.
Paginating it does not help; fetching the whole superfamily this way would need ~84
sequential ~38s requests, worse than the single unpaginated call.

`/asserted_distributions?descendants=true` (what `useKeyGeography.js` already uses for
genus-and-above terminals) is ~8× faster per page and constant regardless of page
position; its cost is purely the **page count** (~50–54 pages for all of
Curculionoidea), which is why key 5024 is slow to load but not as slow as a `dwc.json`
call would be.

See `docs/feasibility_key_geography_filter.md` and the `question` file for the
conversation this benchmark supports.

## Endpoints used in the geography-scope git history

`git log --oneline --all -- <file>` for every file the feature touched, in
chronological order (oldest first), with the endpoint/param shape each commit
introduced or changed. `/dwc_occurrences` (the direct REST endpoint benchmarked
above) has **never** been called from this codebase — only the `/otus/:id/inventory/dwc.json`
wrapper around it, and always without `page`/`per` until this session.

| Commit | File(s) | What changed |
|---|---|---|
| `f25a722` feat(keys): filter a dichotomous key by geography | `useKeyGeography.js`, `geoNormalize.js` | First version. `GET /asserted_distributions?otu_id[]=...` (direct only) + `GET /otus/:id/inventory/dwc.json` per terminal, no pagination. |
| `67a1317` feat(keys): geography filter rolls up the decision path; resolve higher-rank terminals | `useKeyGeography.js`, `geoMatch.js` | Adds `GET /otus?otu_id[]=` + `GET /taxon_names?taxon_name_id[]=` (rank resolution), and `GET /asserted_distributions?taxon_name_id[]=&descendants=true` for higher-rank terminals. |
| `526b8fc` fix(keys): join asserted distributions by asserted_distribution_object, split completeness chips | `useKeyGeography.js` | Same endpoints; fixes how AD rows are joined to their OTU. |
| `ae017a4` fix(keys): address code-review findings on the geography filter | `useKeyGeography.js` | Same endpoints; introduces the `fetchAllAD` pager (follows `/asserted_distributions` pages until a short one). |
| `84d67aa` fix(keys): drop BiologicalAssociation ADs from the picker counts; reorder stats chips | `useKeyGeography.js` | Same endpoints; filters out non-OTU asserted-distribution rows. |
| `b657b5d` fix(keys): geo completeness reuses the picker's data; filter non-Otu ADs | `useKeyGeography.js`, `completeness.js` | Same endpoints; completeness pass reuses the picker's fetched data instead of re-fetching. |
| `417cc8e` fix(keys): make geography scoping viable for higher-taxon terminals | `useKeyGeography.js`, `geoScope.js` | Adds the `needsDescendantAd` / `needsSpecimenPass` gating (this notebook's `geoScope.js` reference) so `dwc.json` is only called for terminals cheap enough to afford it — family/tribe/giant-genus terminals skip it entirely. Comment already flags `/inventory/dwc.json` as the expensive one relative to `/asserted_distributions`. |
| `2566b90` fix: address code-review findings on the geo + citation changes | `useKeyGeography.js` | Current state (this session's starting point): batches `/asserted_distributions` pages (`AD_PAGE_CONCURRENCY`), still calls `dwc.json` with no `page`/`per`. |
| `b469f9c` fix(keys): don't read a sub-national area named like a country as that country | `geoNormalize.js` | Unrelated to API shape — country-string normalization only. |

Full endpoint inventory as of `2566b90` (current `setup` state):
- `GET /otus?otu_id[]=` — terminal OTU → taxon_name_id
- `GET /taxon_names?taxon_name_id[]=` — taxon_name → rank
- `GET /asserted_distributions?otu_id[]=` — direct assertions on a terminal
- `GET /asserted_distributions?taxon_name_id[]=&descendants=true` — rollup for genus-and-above terminals
- `GET /otus/:id/inventory/dwc.json` (no `page`/`per`) — specimen countries, gated to cheap terminals only